# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge: Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and analyze the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if it isn't already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Inspect the available record sets and their fields. All entities will be referenced by their `@id` for reproducibility and clarity.

Let's list all available record set IDs and their associated field IDs.

In [ ]:
# List all record set @ids, field @ids, and column @ids in the dataset
print("Available Record Sets and their fields/columns:")
record_sets = [rs for rs in dataset.record_sets]
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    # List fields
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for f in fields:
            print(f"    - Field @id: {f['@id']} (name: {f.get('name', 'N/A')})")
            # List columns within the field
            if 'column' in f:
                columns = f['column']
                if not isinstance(columns, list):
                    columns = [columns]
                for col in columns:
                    # col could be an @id string or a dict
                    if isinstance(col, dict) and '@id' in col:
                        print(f"        - Column @id: {col['@id']} (name: {col.get('name', 'N/A')})")
                    elif isinstance(col, str):
                        print(f"        - Column @id: {col}")

## 3. Data Extraction
Now we will extract records from each record set and load them into Pandas DataFrames. Remember to use `@id` values when specifying record sets, fields, or columns.

In [ ]:
# Gather all RecordSet @ids for extraction
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("All detected RecordSet @ids:")
for rid in all_record_set_ids:
    print(f"  - {rid}")

# Let's load data for every record set
dataframes = {}
for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if len(df) > 0:
            print(f"Loaded {len(df)} records from RecordSet {rs_id}.")
        else:
            print(f"No records found for RecordSet {rs_id}.")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"Error loading RecordSet {rs_id}: {e}")

# For illustration, let's pick the first non-empty RecordSet for further analysis
main_rs_id = None
for rid, df in dataframes.items():
    if isinstance(df, pd.DataFrame) and len(df) > 0:
        main_rs_id = rid
        break
if main_rs_id is not None:
    print(f"Main analysis will use RecordSet @id: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No non-empty record set found to analyze.")

## 4. Exploratory Data Analysis (EDA)
Let's process the data: filter by a numeric field, normalize, remove outliers, and group by a categorical field. 

All references are by `@id`. Adjust the `numeric_field_id` and `group_field_id` below based on the DataFrame columns from the previous section.

In [ ]:
# If the main_rs_id is not set or there are no columns, skip EDA
if main_rs_id is not None and len(dataframes[main_rs_id].columns) > 0:
    df = dataframes[main_rs_id]
    print("Columns available for EDA:", df.columns.tolist())
    
    # Attempt to select a numeric field (by @id, i.e., column name)
    # You may need to adjust the following line depending on your specific dataset
    # Here, as a demonstration, we select the first numeric-like column
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        # Try convert to numeric, see if any values convert
        try:
            values = pd.to_numeric(df[col], errors='coerce')
            if values.notnull().sum() > 0:
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"Using numeric field: {numeric_field_id}")

        # Set threshold dynamically (10th percentile to avoid empty filter)
        threshold = df[numeric_field_id].astype(float).quantile(0.1)
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        mean = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        std = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a likely categorical field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df) / 2:
                group_field_id = col
                break
        if group_field_id is not None:
            print(f"Grouping by {group_field_id}:")
            group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(group_means.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in DataFrame.")
else:
    print("No suitable data for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized variant. If a group field was found, show distributions by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and 'numeric_field_id' in locals() and numeric_field_id is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12,4))

    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), ax=ax[0], kde=True)
    ax[0].set_title(f'Distribution of {numeric_field_id}')
    
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), ax=ax[1], kde=True)
    ax[1].set_title(f'Normalized {numeric_field_id}')
    plt.tight_layout()
    plt.show()

    # If grouping field found, show boxplot
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field_id], y=pd.to_numeric(filtered_df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print("No data to visualize.")

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR² ordered logistic regression dataset using the `mlcroissant` library.

Key steps included identifying record sets and fields by `@id`, extracting tabular data, performing basic EDA (value filtering, normalization, grouping), and producing visualizations to better understand field distributions and relationships.

You can further enrich this analysis by customizing filtering/grouping logic based on the full Croissant schema metadata and investigating dataset-specific insights, such as household predictors for knowledge adoption or regional differences.